# 🖥️ CUDA 베이스라인 (Colab/Runpod) — RT-DETR · Cascade R-CNN · DINO

MPS로 학습 불가한 모델을 **동일 fold0 번들**로 CUDA에서 돌린다. MPS 5종과 **같은 mAP@[0.75:0.95] 하니스**를 써서 결과를 그대로 합류.

## 실행 방법
1. **런타임 → GPU** 로 변경(Colab: 런타임 유형 변경 → T4/GPU).
2. `lhk_cuda_bundle.zip` 을 업로드(왼쪽 파일창 드래그) — 또는 Google Drive에 두고 마운트.
3. 위에서부터 셀 실행. **RT-DETR 먼저**(가장 안정) → mmdet(Cascade/DINO)는 설치 버전 이슈가 있을 수 있어 best-effort.
4. 완료 후 `cuda_baselines.json` 을 다운로드해 KAI에게 전달 → MPS 표에 합류.

## 0. 셋업 · 번들 언집 · 공통 하니스

In [ ]:
import os, zipfile
from pathlib import Path
import torch
BUNDLE_ZIP = "lhk_cuda_bundle.zip"        # 업로드한 파일명 (Drive면 경로 수정)
ROOT = Path("/content/lhk_cuda_bundle")
if not ROOT.exists():
    with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall("/content")
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(GPU 런타임으로 변경 필요!)")
# data.yaml 절대경로 보정 (ultralytics용)
dy = (ROOT/"data.yaml").read_text()
if "path: ." in dy: (ROOT/"data.yaml").write_text(dy.replace("path: .", f"path: {ROOT}"))
print("bundle:", sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# 공통 mAP@[0.75:0.95] 하니스 (MPS와 동일) — 예측 dts는 category_id=dl_idx, image_id=val 정렬순 1..51
import json, numpy as np
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
m2c = {int(k): v for k, v in json.load(open(ROOT/"class_map.json"))["model_index_to_category_id"].items()}
cocoGt = COCO(str(ROOT/"val_gt.json"))
val_imgs = sorted((ROOT/"images/val").glob("*.png"))
nid = {p.name: i+1 for i, p in enumerate(val_imgs)}
RESULTS = []
def score(name, dts, params_M):
    if not dts:
        mAP = ap75 = 0.0
    else:
        e = COCOeval(cocoGt, cocoGt.loadRes(dts), "bbox"); e.params.iouThrs = np.linspace(0.75, 0.95, 5)
        e.evaluate(); e.accumulate(); e.summarize(); mAP = float(e.stats[0])
        e2 = COCOeval(cocoGt, cocoGt.loadRes(dts), "bbox"); e2.params.iouThrs = np.array([0.75])
        e2.evaluate(); e2.accumulate(); e2.summarize(); ap75 = float(e2.stats[0])
    RESULTS.append({"model": name, "params_M": round(params_M, 1), "mAP_75_95": round(mAP, 4), "ap75": round(ap75, 4), "val_det": len(dts)})
    json.dump(RESULTS, open(ROOT/"cuda_baselines.json", "w"), ensure_ascii=False, indent=2)
    print(f">>> {name}: mAP@[0.75:0.95]={mAP:.4f}  AP@0.75={ap75:.4f}  val_det={len(dts)}")

## 1. RT-DETR (ultralytics) — 안정적, 먼저 실행

In [ ]:
!pip -q install ultralytics
from ultralytics import RTDETR
m = RTDETR("rtdetr-l.pt")
m.train(data=str(ROOT/"data.yaml"), epochs=50, imgsz=640, batch=8, device=0, seed=42,
        project=str(ROOT/"runs"), name="rtdetr", exist_ok=True, plots=False, verbose=False)
best = RTDETR(str(ROOT/"runs/rtdetr/weights/best.pt"))
dts = []
for p in val_imgs:
    r = best.predict(str(p), conf=0.001, iou=0.6, max_det=20, device=0, verbose=False)[0]
    for b, c, s in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
        x1, y1, x2, y2 = b
        dts.append({"image_id": nid[p.name], "category_id": m2c[int(c)], "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)], "score": float(s)})
score("RT-DETR-l", dts, sum(pp.numel() for pp in best.model.parameters())/1e6)

## 1b. Faster R-CNN (torchvision, warmup) — MPS서 RoIAlign CPU폴백으로 너무 느려(에폭 52분) CUDA로 이관
발산 방지: **선형 warmup + grad-clip**. (MPS 표의 0.0은 warmup 부재 발산 → 여기서 진짜 수치 산출)

In [ ]:
import torch, time
from PIL import Image
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
Wp, Hp = 976, 1280

class DS(Dataset):
    def __init__(self, sp): self.im = sorted((ROOT/"images"/sp).glob("*.png")); self.sp = sp
    def __len__(self): return len(self.im)
    def __getitem__(self, i):
        p = self.im[i]; img = Image.open(p).convert("RGB"); bx = []; lb = []
        for ln in (ROOT/"labels"/self.sp/(p.stem+".txt")).read_text().splitlines():
            if not ln.strip(): continue
            m, cx, cy, nw, nh = ln.split(); m = int(m); cx, cy, nw, nh = map(float, (cx, cy, nw, nh))
            bx.append([(cx-nw/2)*Wp, (cy-nh/2)*Hp, (cx+nw/2)*Wp, (cy+nh/2)*Hp]); lb.append(m+1)
        return TF.to_tensor(img), {"boxes": torch.tensor(bx, dtype=torch.float32), "labels": torch.tensor(lb, dtype=torch.int64)}
def coll(b): return tuple(zip(*b))

mdl = fasterrcnn_resnet50_fpn(weights="DEFAULT")
mdl.roi_heads.box_predictor = FastRCNNPredictor(mdl.roi_heads.box_predictor.cls_score.in_features, 57)
mdl.transform.min_size = (512,); mdl.transform.max_size = 800; mdl.cuda()
dl = DataLoader(DS("train"), batch_size=4, shuffle=True, collate_fn=coll, num_workers=2)
opt = torch.optim.SGD([p for p in mdl.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=5e-4)
BL, EP = 0.005, 24; wu = min(500, len(dl)*2); g = 0; t0 = time.time()
for ep in range(EP):
    mdl.train(); tot = 0.0
    for im, tg in dl:
        g += 1
        if g <= wu:
            for pg in opt.param_groups: pg["lr"] = BL*(0.01+0.99*g/wu)
        im = [x.cuda() for x in im]; tg = [{k: v.cuda() for k, v in t.items()} for t in tg]
        loss = sum(mdl(im, tg).values())
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(mdl.parameters(), 10); opt.step(); tot += float(loss)
    if ep >= int(EP*0.8):
        for pg in opt.param_groups: pg["lr"] = BL*0.1
    print(f"ep{ep+1}/{EP} loss={tot/len(dl):.3f} ({time.time()-t0:.0f}s)")

mdl.eval(); dts = []
with torch.no_grad():
    for p in val_imgs:
        o = mdl([TF.to_tensor(Image.open(p).convert("RGB")).cuda()])[0]
        for b, l, sc in zip(o["boxes"].cpu().numpy(), o["labels"].cpu().numpy(), o["scores"].cpu().numpy()):
            if sc < 0.001: continue
            idx = int(l) - 1
            if idx not in m2c: continue
            x1, y1, x2, y2 = b
            dts.append({"image_id": nid[p.name], "category_id": m2c[idx], "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)], "score": float(sc)})
score("FasterRCNN-R50(warmup)", dts, sum(q.numel() for q in mdl.parameters())/1e6)

## 2. Cascade R-CNN · DINO (mmdet) — ⛔ 이 Colab에선 보류(건너뛰기)

**환경 제약으로 실행 불가:** ① `openmim→openxlab`가 Python 3.12와 비호환(`pkgutil.ImpImporter` 제거) · ② `mmcv` prebuilt wheel이 torch 2.11+cu128에 없음(소스빌드 실패). 
→ **아래 2·3·4 셀은 건너뛰고**, 위 `1. RT-DETR` + `1b. FasterRCNN` 결과만 사용. (전 계열 커버리지는 MPS 4종 + 이 둘로 충분.) 
DINO 대체가 꼭 필요하면 **HF `Deformable-DETR`**(mmcv 불필요, 순수 PyTorch) 셀을 요청하세요.

In [ ]:
# mmdet 설치 (Colab Python 3.12 대응) — setuptools 먼저 고쳐 openmim/pkg_resources(ImpImporter) 에러 회피
import sys, torch; print("Python", sys.version.split()[0], "| torch", torch.__version__, "| cuda", torch.version.cuda)
!pip -q install -U pip "setuptools>=70" wheel
!pip -q install -U openmim
!mim install -q mmengine
!mim install -q "mmcv>=2.1.0,<2.3.0"
!mim install -q "mmdet>=3.3.0"
import mmengine, mmcv, mmdet
print("OK mmdet", mmdet.__version__, "| mmcv", mmcv.__version__)
# 그래도 실패하면: 런타임 → 런타임 다시 시작 후 이 셀 재실행. mmcv 소스빌드가 계속 막히면 Cascade/DINO는 건너뛰고 RT-DETR·FasterRCNN만 사용.

In [ ]:
# mmdet 학습 헬퍼: base config를 우리 fold0 COCO(56클래스)로 override → 학습 → val 추론 → 하니스 채점
from mmengine.config import Config
from mmengine.runner import Runner

CLASSES = tuple(str(m2c[i]) for i in range(56))
META = dict(classes=CLASSES)

def set_num_classes(model_cfg, nc=56):
    rh = model_cfg.get("roi_head")
    if rh is not None:                       # Cascade/Faster: roi_head.bbox_head (list 또는 dict)
        bh = rh["bbox_head"]
        for h in (bh if isinstance(bh, list) else [bh]): h["num_classes"] = nc
    if model_cfg.get("bbox_head") is not None:   # DINO/DETR 계열
        model_cfg["bbox_head"]["num_classes"] = nc

def run_mmdet(cfg_path, run_name, epochs=24, base_lr=None):
    cfg = Config.fromfile(cfg_path)
    for split, ann, pref in [("train", "coco/train.json", "images/train/"), ("val", "coco/val.json", "images/val/")]:
        dl = cfg[f"{split}_dataloader"]
        dl["dataset"].update(data_root=str(ROOT), ann_file=ann, data_prefix=dict(img=pref), metainfo=META)
    cfg.train_dataloader["batch_size"] = 2
    cfg.test_dataloader = cfg.val_dataloader
    cfg.val_evaluator.update(ann_file=str(ROOT/"coco/val.json"), metric="bbox"); cfg.test_evaluator = cfg.val_evaluator
    set_num_classes(cfg.model, 56)
    cfg.train_cfg["max_epochs"] = epochs
    if base_lr is not None: cfg.optim_wrapper["optimizer"]["lr"] = base_lr
    cfg.default_hooks["checkpoint"].update(interval=epochs, save_best="coco/bbox_mAP")
    cfg.work_dir = str(ROOT/f"runs/{run_name}")
    cfg.load_from = None  # 필요시 COCO 사전학습 ckpt URL 지정
    runner = Runner.from_cfg(cfg); runner.train()
    return runner, cfg

def mmdet_eval(run_name, cfg, label):
    from mmdet.apis import init_detector, inference_detector
    import glob
    ck = sorted(glob.glob(str(ROOT/f"runs/{run_name}/best_*.pth")) or glob.glob(str(ROOT/f"runs/{run_name}/epoch_*.pth")))[-1]
    model = init_detector(cfg, ck, device="cuda:0")
    dts = []
    for p in val_imgs:
        res = inference_detector(model, str(p)).pred_instances
        bb = res.bboxes.cpu().numpy(); lb = res.labels.cpu().numpy(); sc = res.scores.cpu().numpy()
        for (x1, y1, x2, y2), l, s in zip(bb, lb, sc):
            if s < 0.001: continue
            dts.append({"image_id": nid[p.name], "category_id": m2c[int(l)], "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)], "score": float(s)})
    pm = sum(q.numel() for q in model.parameters())/1e6
    score(label, dts, pm)

## 3. Cascade R-CNN (고-IoU 정합 2-stage)

In [ ]:
r, c = run_mmdet("mmdet::cascade_rcnn/cascade-rcnn_r50_fpn_1x_coco.py", "cascade", epochs=24)
mmdet_eval("cascade", c, "Cascade-RCNN-R50")

## 4. DINO (정확도 트랜스포머) — 수렴 위해 에폭↑ 권장

In [ ]:
r, c = run_mmdet("mmdet::dino/dino-4scale_r50_8xb2-12e_coco.py", "dino", epochs=36)
mmdet_eval("dino", c, "DINO-4scale-R50")

## 5. DINOv2-frozen 백본 (선택·심화)
`torch.hub.load('facebookresearch/dinov2','dinov2_vits14')` 를 frozen 특징추출기로 쓰고 경량 검출 헤드를 얹는 방식. 배선이 커서 시간 여유 시 별도 진행(여기선 생략).

## 6. 결과 요약 · 내보내기

In [ ]:
import json
from pathlib import Path
ROOT = Path("/content/lhk_cuda_bundle")
f = ROOT / "cuda_baselines.json"                      # score()가 매번 저장하는 파일
rows = json.load(open(f)) if f.exists() else globals().get("RESULTS", [])
if not rows:
    print("⚠️ 결과 비어있음 — 0(셋업)→1(RT-DETR)→1b(FasterRCNN) 순서로 실행됐는지,")
    print("   그 셀들이 '>>> ... mAP@[0.75:0.95]=...' 를 출력했는지 확인하세요.")
else:
    import pandas as pd
    print(pd.DataFrame(rows).sort_values("mAP_75_95", ascending=False).to_string(index=False))
    try:
        from google.colab import files; files.download(str(f))
    except Exception:
        print("\n수동 다운로드: 왼쪽 파일창에서", f)